# Streaming Live Events

screamer's functors serve two workflows from a single implementation.
In **backtest mode** you pass an entire NumPy array and get an array back -
fast and convenient for offline analysis.  In **live mode** you feed the
same functor one scalar at a time (e.g. from a socket or a message queue)
and collect results incrementally.  Because both paths share the same C++
state machine, the numbers are guaranteed to be identical: no train/serve
skew.

In [ ]:
import numpy as np
from screamer import RollingZscore

rng = np.random.default_rng(1)
x = rng.standard_normal(500)

## Batch path - the backtest

Pass the whole array at once.  The functor processes all 500 samples in a
single vectorised C++ call and returns a (500,) array.

In [ ]:
# BATCH: whole array at once (the backtest path)
batch = RollingZscore(50)(x)
print("batch result shape:", batch.shape)
print("first non-NaN value:", batch[~np.isnan(batch)][0])

## Live path - one event at a time

The same functor class is instantiated once, then called with a single float
on each event.  The call is O(1): the functor maintains just enough internal
state to produce the next output without re-scanning history.

In [ ]:
# LIVE: feed one event at a time (the production path) - same functor class
z = RollingZscore(50)
live = []
for v in x:                        # imagine v arriving from a socket / queue
    live.append(z(v))              # scalar in, scalar out; state persists

print("live result length:", len(live))
print("first non-NaN value:", next(v for v in live if not np.isnan(v)))

## Batch == live (the key guarantee)

The two paths must produce byte-identical results.  The in-cell assertion
below is the regression test: if it ever fails, screamer has a causality bug.

In [ ]:
np.testing.assert_array_equal(batch, np.array(live))
print("backtest == live:", np.allclose(batch, live, equal_nan=True))

## Back-pressure and lazy iteration

For generator-based live sources you can also pass an iterator directly.
screamer wraps it in a `LazyIterator` that pulls one value at a time -
results come out as you consume them, so memory stays bounded even for
infinite streams.

**Important:** always bind the functor to a named variable before passing
an iterator to it.  A temporary functor can be garbage-collected while the
lazy iterator still holds a reference, causing a segfault.

In [ ]:
def live_source(vals):
    """Simulate a live data source yielding one float at a time."""
    for v in vals:
        yield v

z2 = RollingZscore(50)                        # named - keeps functor alive
lazy_result = list(z2(live_source(x)))        # iterator in, iterator out

np.testing.assert_array_equal(batch, np.array(lazy_result))
print("lazy iterator result also equals batch ✓")

**Takeaways**

- Every screamer functor supports both batch (array) and live (scalar) calling.
- The scalar call is O(1) per event - the functor holds only the minimal state
  it needs, not the full history.
- Batch and live results are bit-for-bit identical: no train/serve skew by
  construction.
- Always assign the functor to a named variable before feeding it an iterator
  (prevents premature garbage collection of the C++ object).